# 単語の共起を数えて PMI で結びつきの強さを測る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/05_corpus_statistics/pmi.ipynb)

NLP クックブック 05-09 ／ コーパスの統計と可視化

同じ文に一緒に出てきやすい単語の組を見つけたい。単なる頻度ではなく、偶然より多く共起しているかで測りたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, re, urllib.request, zipfile
from collections import Counter
from itertools import combinations
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

text = aozora("https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip")
tagger = Tagger()
sentences = [s for s in re.split(r"[。\n]", text) if s.strip()]
docs = [{w.feature.orthBase or w.surface for w in tagger(s)
         if w.feature.pos1 in ("名詞", "動詞", "形容詞") and w.feature.pos2 not in ("非自立可能", "数詞")} for s in sentences]
print(len(docs), "文")

9069 文


In [3]:
import math

N = len(docs)
word_count = Counter(w for d in docs for w in d)
pair_count = Counter(pair for d in docs for pair in combinations(sorted(d), 2))

def pmi(a, b):
    a, b = sorted((a, b))
    return math.log2(pair_count[a, b] * N / (word_count[a] * word_count[b]))

print("共起頻度の上位:", pair_count.most_common(8))
strong = sorted(((pmi(a, b), a, b, c) for (a, b), c in pair_count.items() if c >= 10), reverse=True)
for score, a, b, c in strong[:15]:
    print(f"{score:5.2f}  {a} - {b}  (共起 {c} 回)")

共起頻度の上位: [(('事', '云う'), 284), (('もの', '云う'), 193), (('主人', '云う'), 180), (('もの', '事'), 151), (('主人', '事'), 130), (('もの', '主人'), 108), (('云う', '迷亭'), 96), (('沙弥', '苦'), 95)]
 9.82  呂 - 崎  (共起 10 回)
 9.69  サルト - デル  (共起 11 回)
 9.69  アンドレア - デル  (共起 11 回)
 9.69  アンドレア - サルト  (共起 11 回)
 9.56  琴 - 絃  (共起 10 回)
 8.82  け出す - 馳  (共起 10 回)
 8.73  大和 - 魂  (共起 14 回)
 8.68  天然 - 居士  (共起 18 回)
 8.43  御免 - 蒙る  (共起 11 回)
 8.10  右衛門 - 武  (共起 33 回)
 8.04  新体 - 詩  (共起 13 回)
 8.01  意気 - 生  (共起 14 回)
 7.85  巻 - 煙草  (共起 10 回)
 7.61  折れる - 骨  (共起 15 回)
 7.45  側 - 椽  (共起 37 回)


- PMI は低頻度語の組を過大評価するので、共起頻度の下限（ここでは 10）を設けるのが必須。
- 負の値を 0 に切り上げた PPMI を成分とする共起行列は、古典的な単語ベクトルになる。